# Local OCR

In [ ]:
import os

# Chemin du notebook → racine du projet
repo_root = os.path.dirname(os.path.abspath("__file__"))
requirements = os.path.join(repo_root, "requirements.txt")

%pip install -r "{requirements}"

In [1]:
import fitz
import easyocr
from PIL import Image
import io
import numpy as np

In [33]:
def ocr_pdf(pdf_path: str, languages: list = ['fr']) -> str:
    # 1. Ouvrir le PDF
    doc = fitz.open(pdf_path)
    reader = easyocr.Reader(languages)
    full_text = []

    for page_num, page in enumerate(doc):
        # 2. Convertir la page en image (300 DPI recommandé)
        mat = fitz.Matrix(300 / 72, 300 / 72)  # scale pour 300 DPI
        pix = page.get_pixmap(matrix=mat)
        img_bytes = pix.tobytes("png")

        # 3. OCR sur l'image
        img = Image.open(io.BytesIO(img_bytes))
        img_np = np.array(img)          # ← conversion PIL → numpy
        result = reader.readtext(img_np, detail=0)
        
        torch.mps.empty_cache()

        full_text.append(f"--- Page {page_num + 1} ---")
        full_text.append("\n".join(result))

    doc.close()
    return "\n".join(full_text)

In [ ]:
def extract_text_native(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    result = {}
    result["OCRResult"] = ocr_pdf(pdf_path)
    result["TextExtract"] = extract_text_native(pdf_path)
    return result

# Mistral OCR

In [ ]:
import base64
import os
from mistralai.client import Mistral

In [ ]:
load_dotenv()
api_key = os.getenv("MISTRAL_API_KEY")
client = Mistral(api_key=api_key)

In [ ]:
def encode_pdf(pdf_path):
    with open(pdf_path, "rb") as pdf_file:
        return base64.b64encode(pdf_file.read()).decode('utf-8')

pdf_path = "path_to_your_pdf.pdf"
base64_pdf = encode_pdf(pdf_path)

ocr_response = client.ocr.process(
    model="mistral-ocr-latest",
    document={
        "type": "document_url",
        "document_url": f"data:application/pdf;base64,{base64_pdf}"
    },
    table_format="html", # default is None
    # extract_header=True, # default is False
    # extract_footer=True, # default is False
    include_image_base64=True
)

# Text to File Name

In [ ]:
import os
from mistralai.client import Mistral
from dotenv import load_dotenv

In [ ]:
def text_to_file_name(text: str, api_key: str) -> str:
    model = "mistral-small-latest"

    client = Mistral(api_key=api_key)

    chat_response = client.chat.complete(
        model = model,
        messages = [
            {
                "role": "system",
                "content": (
                    "Je vais te fournir le résultat d'un scan de facture.\n"
                    "C'est un JSON avec 2 clefs : OCRResult et TextExtract. Le premier est le résultat d'un OCR, le deuxième est le texte natif du pdf.\n\n"
                    "Je veux que tu extraies 3 informations de cela :\n"
                    "1. L'établissement auquel est destinée la facture. Il y a 5 choix possibles :\n"
                    "   - Eglantines (ELGT)\n"
                    "   - Accueil Temporaire (AT)\n"
                    "   - Foyer de Vie (FV)\n"
                    "   - Beguinage (BEG)\n"
                    "   - Association Gérontologique de l'Estuaire (AGE)\n"
                    "2. L'entreprise qui a émis la facture\n"
                    "3. La date de facture (date à laquelle la facture est émise)\n\n"
                    "Je veux alors que tu me retournes ces informations sous le format ETABLISSEMENT-EMETTEUR-DATE.\n"
                    "Voici des exemples :\n"
                    " - EGL-Essity-2026-03-13\n"
                    " - AT-Essity-2026-03-13\n"
                    " - FV-Kalhyge-2026-03-13\n"
                    " - BEG-Neosilver-2026-03-13\n"
                    " - AGE-Restologis-2026-03-13\n"
                    "Je ne veux rien d'autre dans le retour. Si tu n'arrives pas à déterminer ces informations, renvoie : \"null\"."
                ),
            },
            {
                "role": "user",
                "content": str(extracted_text),
            },
        ]
    )
    
    return chat_response.choices[0].message.content

In [ ]:
file_name = text_to_file_name(extracted_text)
print(file_name)

# Rename Invoice

In [ ]:
def rename_invoice(pdf_path: str, new_name: str):
    os.rename(pdf_path, new_name)


# Full Process

In [ ]:
import os
import torch  # Pour empty_cache()
from easyocr import Reader
import os

In [ ]:
def process_single_pdf(folder_path, file):
    """Fonction pour traiter un seul PDF (appelée dans un subprocess)"""
    print(f"Processing file: {file}")
    pdf_path = os.path.join(folder_path, file)
    extracted_text = extract_text_from_pdf(pdf_path)
    file_name = text_to_file_name(extracted_text)
    rename_invoice(pdf_path, file_name + ".pdf")
    print(f"File {os.path.basename(pdf_path)} renamed to {file_name}.pdf")

In [ ]:
folder_path = "./Samples"
reader = Reader(['fr', 'en'])  # Initialise UNE SEULE FOIS

for file in os.listdir(folder_path):
    if file.endswith(".pdf"):
        process_single_pdf(folder_path, file)
